In [2]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(scutilsR)
library(tidyverse)
library(celda)
library(sceasy)
library(Nebulosa)
library(glue)

In [3]:
seu = qs::qread("/home/data/tanglei/project/prostate_altas/output/04/QC_fin.qs")

In [ ]:
## miloR + Augur: type 一对多 (Target vs Other) × H2/H3
library(miloR)
library(SingleCellExperiment)
library(scater)
library(scran)
library(dplyr)
library(ggplot2)
library(Augur)
library(ggrepel)

out_dir <- "/home/data/tanglei/project/prostate_altas/output/04/miloR+AuguR"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

types <- c("5aTreat_BPH", "Cribriform_PC", "CRPC_A", "Naive_BPH", "Naive_PC", "NEPC", "Normal")
ct_cols <- c("celltype_manual_H2", "celltype_manual_H3")

## 子集到 7 类
seu_sub <- subset(seu, subset = type %in% types)
message("Cells after subset: ", ncol(seu_sub))
print(table(seu_sub$type))

## 一次性构建 milo 图结构（与 celltype / target 无关）
rpca.embeddings <- Embeddings(seu_sub, reduction = "rpca")
umap.embeddings <- Embeddings(seu_sub, reduction = "umap.rpca")

sce <- SingleCellExperiment(
  assays = list(counts = GetAssayData(seu_sub, assay = "RNA", layer = "counts")),
  reducedDims = SimpleList(RPCA = rpca.embeddings, UMAP = umap.embeddings)
)

milo_base <- Milo(sce)
milo_base <- buildGraph(milo_base, k = 30, d = 30, reduced.dim = "RPCA")
milo_base <- makeNhoods(milo_base, prop = 0.1, k = 30, d = 30, refined = TRUE, reduced_dims = "RPCA")
milo_base <- calcNhoodDistance(milo_base, d = 30, reduced.dim = "RPCA")
message("milo graph ready")

## 双层循环：celltype 层级 × target
for (ct_col in ct_cols) {
  ct_level <- ifelse(grepl("H2$", ct_col), "H2", "H3")
  message("==== celltype: ", ct_col, " (", ct_level, ") ====")

  for (target in types) {
    message("-- target: ", target, " vs Other | ", ct_col)

    tryCatch({
      ## 二分类标签
      group_vec <- ifelse(seu_sub$type == target, "Target", "Other")
      group_vec <- factor(group_vec, levels = c("Other", "Target"))
      sample_vec <- paste0(seu_sub$orig.ident, ".", as.character(group_vec))
      celltype_vec <- as.character(seu_sub[[ct_col, drop = TRUE]])

      meta.data <- data.frame(
        sample = sample_vec,
        group = group_vec,
        celltype = celltype_vec,
        row.names = colnames(seu_sub),
        stringsAsFactors = FALSE
      )

      milo <- milo_base
      colData(milo) <- DataFrame(meta.data)
      milo <- countCells(milo, meta.data = as.data.frame(colData(milo)), sample = "sample")

      milo_design <- as.data.frame(colData(milo)) %>%
        dplyr::select(sample, group) %>%
        distinct()
      milo_design$group <- factor(milo_design$group, levels = c("Other", "Target"))
      rownames(milo_design) <- milo_design$sample

      da_results <- testNhoods(milo, design = ~ group, design.df = milo_design, reduced.dim = "RPCA")
      da_results <- annotateNhoods(milo, da_results, coldata_col = "celltype")
      da_results$celltype <- ifelse(da_results$celltype_fraction < 0.7, "Mixed", da_results$celltype)

      plot_title <- paste0(target, " vs Other | ", ct_col)

      p_milo <- plotDAbeeswarm(da_results, group.by = "celltype", alpha = 1) +
        scale_color_gradient2(midpoint = 0, low = "blue", mid = "white", high = "red", space = "Lab") +
        geom_hline(yintercept = 0, linetype = "dashed") +
        ggtitle(plot_title)
      ggsave(
        file.path(out_dir, paste0("miloR_", ct_level, "_", target, ".pdf")),
        p_milo, width = 8, height = 6
      )

      ## Augur
      seu_sub$augur_label <- as.character(group_vec)
      augur <- calculate_auc(
        seu_sub,
        cell_type_col = ct_col,
        label_col = "augur_label",
        n_threads = 1
      )

      data.plot <- subset(augur$results, metric == "roc_auc")
      data.plot$cell_type <- factor(data.plot$cell_type, levels = rev(augur$AUC$cell_type))
      p_augur_violin <- ggplot(data.plot, aes(cell_type, estimate, color = cell_type)) +
        geom_violin() +
        geom_jitter(size = .3) +
        coord_flip() +
        theme_bw() +
        theme(legend.position = "none") +
        ggtitle(plot_title)
      ggsave(
        file.path(out_dir, paste0("Augur_violin_", ct_level, "_", target, ".pdf")),
        p_augur_violin, width = 7, height = 6
      )

      data.bar <- augur$AUC
      data.bar$cell_type <- factor(data.bar$cell_type, levels = rev(augur$AUC$cell_type))
      p_augur_bar <- ggplot(data.bar, aes(cell_type, auc, fill = cell_type)) +
        geom_col() +
        coord_flip() +
        theme_bw() +
        theme(legend.position = "none") +
        ggsci::scale_fill_d3("category20") +
        ggtitle(plot_title)
      ggsave(
        file.path(out_dir, paste0("Augur_bar_", ct_level, "_", target, ".pdf")),
        p_augur_bar, width = 7, height = 6
      )

      ## Combined (errorbar)
      data1 <- da_results %>%
        group_by(celltype) %>%
        summarise(logFC = mean(logFC), xerr = sd(logFC), .groups = "drop")
      colnames(data1) <- c("cell_type", "logFC", "xerr")

      data2 <- subset(augur$results, metric == "roc_auc") %>%
        group_by(cell_type) %>%
        summarise(auc = mean(estimate), yerr = sd(estimate), .groups = "drop")

      data_plot <- inner_join(data1, data2, by = "cell_type")
      ## Mixed 无 Augur 条目时自然被 drop；NA sd 置 0
      data_plot$xerr[is.na(data_plot$xerr)] <- 0
      data_plot$yerr[is.na(data_plot$yerr)] <- 0

      p_combined <- ggplot(data_plot, aes(x = logFC, y = auc, color = cell_type, label = cell_type)) +
        geom_point(size = 4) +
        geom_errorbar(aes(ymin = auc - yerr, ymax = auc + yerr), width = 0.1, alpha = 0.3) +
        geom_errorbarh(aes(xmin = logFC - xerr, xmax = logFC + xerr), height = 0.1, alpha = 0.3) +
        geom_hline(yintercept = 0.5, linetype = "dashed", color = "black") +
        geom_vline(xintercept = 0, linetype = "dashed", color = "black") +
        ggrepel::geom_text_repel(show.legend = FALSE, color = "black") +
        theme_bw(base_size = 15) +
        labs(x = "logFC by MiloR", y = "AUC score by Augur", title = plot_title) +
        theme(
          legend.position = "none",
          panel.grid.major = element_blank(),
          panel.grid.minor = element_blank()
        )
      ggsave(
        file.path(out_dir, paste0("Combined_", ct_level, "_", target, ".pdf")),
        p_combined, width = 8, height = 7
      )

      message("  done: ", target, " / ", ct_level)
    }, error = function(e) {
      message("  FAILED: ", target, " / ", ct_level, " -> ", conditionMessage(e))
    })
  }
}

message("All finished. PDFs in: ", out_dir)

In [4]:
tmp = subset(seu, subset = type == "Naive_PC")

In [5]:
## miloR + Augur on tmp (Naive_PC): PSA / stage / Gleason × H2/H3
library(miloR)
library(SingleCellExperiment)
library(scater)
library(scran)
library(dplyr)
library(ggplot2)
library(Augur)
library(ggrepel)

out_dir <- "/home/data/tanglei/project/prostate_altas/output/04/miloR+AuguR"
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
ct_cols <- c("celltype_manual_H2", "celltype_manual_H3")

## 三个对比：low/ref 在前，high/effect 在后（logFC>0 = 高危侧富集）
contrasts <- list(
  PSA = list(
    tag = "PSA",
    make_group = function(obj) {
      psa <- suppressWarnings(as.numeric(as.character(obj$PSA)))
      keep <- !is.na(psa)
      group <- ifelse(psa[keep] >= 10, "PSA_high", "PSA_low")
      list(cells = colnames(obj)[keep], group = factor(group, levels = c("PSA_low", "PSA_high")))
    }
  ),
  stage = list(
    tag = "stage",
    make_group = function(obj) {
      st <- as.character(obj$stage)
      low <- c("pT2", "pT2a", "pT2c")
      high <- c("pT3a", "pT3b")
      keep <- st %in% c(low, high)
      group <- ifelse(st[keep] %in% high, "stage_pT3", "stage_pT2")
      list(cells = colnames(obj)[keep], group = factor(group, levels = c("stage_pT2", "stage_pT3")))
    }
  ),
  Gleason = list(
    tag = "Gleason",
    make_group = function(obj) {
      gl <- as.character(obj$Gleason)
      low <- c("3+3", "3+4", "4+3")
      high <- c("3+5", "4+4", "4+5", "5+3", "5+5")
      keep <- gl %in% c(low, high)
      group <- ifelse(gl[keep] %in% high, "Gleason_high", "Gleason_low")
      list(cells = colnames(obj)[keep], group = factor(group, levels = c("Gleason_low", "Gleason_high")))
    }
  )
)

run_milo_augur <- function(obj, group_vec, ct_col, out_prefix, plot_title) {
  ct_level <- ifelse(grepl("H2$", ct_col), "H2", "H3")
  sample_vec <- paste0(obj$orig.ident, ".", as.character(group_vec))
  celltype_vec <- as.character(obj[[ct_col, drop = TRUE]])

  meta.data <- data.frame(
    sample = sample_vec,
    group = group_vec,
    celltype = celltype_vec,
    row.names = colnames(obj),
    stringsAsFactors = FALSE
  )

  rpca.embeddings <- Embeddings(obj, reduction = "rpca")
  umap.embeddings <- Embeddings(obj, reduction = "umap.rpca")
  sce <- SingleCellExperiment(
    assays = list(counts = GetAssayData(obj, assay = "RNA", layer = "counts")),
    reducedDims = SimpleList(RPCA = rpca.embeddings, UMAP = umap.embeddings)
  )

  milo <- Milo(sce)
  colData(milo) <- DataFrame(meta.data)
  milo <- buildGraph(milo, k = 30, d = 30, reduced.dim = "RPCA")
  milo <- makeNhoods(milo, prop = 0.1, k = 30, d = 30, refined = TRUE, reduced_dims = "RPCA")
  milo <- countCells(milo, meta.data = as.data.frame(colData(milo)), sample = "sample")

  milo_design <- as.data.frame(colData(milo)) %>%
    dplyr::select(sample, group) %>%
    distinct()
  milo_design$group <- factor(milo_design$group, levels = levels(group_vec))
  rownames(milo_design) <- milo_design$sample

  milo <- calcNhoodDistance(milo, d = 30, reduced.dim = "RPCA")
  da_results <- testNhoods(milo, design = ~ group, design.df = milo_design, reduced.dim = "RPCA")
  da_results <- annotateNhoods(milo, da_results, coldata_col = "celltype")
  da_results$celltype <- ifelse(da_results$celltype_fraction < 0.7, "Mixed", da_results$celltype)

  p_milo <- plotDAbeeswarm(da_results, group.by = "celltype", alpha = 1) +
    scale_color_gradient2(midpoint = 0, low = "blue", mid = "white", high = "red", space = "Lab") +
    geom_hline(yintercept = 0, linetype = "dashed") +
    ggtitle(plot_title)
  ggsave(file.path(out_dir, paste0("miloR_", ct_level, "_", out_prefix, ".pdf")),
         p_milo, width = 8, height = 6)

  obj$augur_label <- as.character(group_vec)
  augur <- calculate_auc(obj, cell_type_col = ct_col, label_col = "augur_label", n_threads = 1)

  data.plot <- subset(augur$results, metric == "roc_auc")
  data.plot$cell_type <- factor(data.plot$cell_type, levels = rev(augur$AUC$cell_type))
  p_augur_violin <- ggplot(data.plot, aes(cell_type, estimate, color = cell_type)) +
    geom_violin() + geom_jitter(size = .3) + coord_flip() +
    theme_bw() + theme(legend.position = "none") + ggtitle(plot_title)
  ggsave(file.path(out_dir, paste0("Augur_violin_", ct_level, "_", out_prefix, ".pdf")),
         p_augur_violin, width = 7, height = 6)

  data.bar <- augur$AUC
  data.bar$cell_type <- factor(data.bar$cell_type, levels = rev(augur$AUC$cell_type))
  p_augur_bar <- ggplot(data.bar, aes(cell_type, auc, fill = cell_type)) +
    geom_col() + coord_flip() + theme_bw() + theme(legend.position = "none") +
    ggsci::scale_fill_d3("category20") + ggtitle(plot_title)
  ggsave(file.path(out_dir, paste0("Augur_bar_", ct_level, "_", out_prefix, ".pdf")),
         p_augur_bar, width = 7, height = 6)

  data1 <- da_results %>%
    group_by(celltype) %>%
    summarise(logFC = mean(logFC), xerr = sd(logFC), .groups = "drop")
  colnames(data1) <- c("cell_type", "logFC", "xerr")
  data2 <- subset(augur$results, metric == "roc_auc") %>%
    group_by(cell_type) %>%
    summarise(auc = mean(estimate), yerr = sd(estimate), .groups = "drop")
  data_plot <- inner_join(data1, data2, by = "cell_type")
  data_plot$xerr[is.na(data_plot$xerr)] <- 0
  data_plot$yerr[is.na(data_plot$yerr)] <- 0

  p_combined <- ggplot(data_plot, aes(x = logFC, y = auc, color = cell_type, label = cell_type)) +
    geom_point(size = 4) +
    geom_errorbar(aes(ymin = auc - yerr, ymax = auc + yerr), width = 0.1, alpha = 0.3) +
    geom_errorbarh(aes(xmin = logFC - xerr, xmax = logFC + xerr), height = 0.1, alpha = 0.3) +
    geom_hline(yintercept = 0.5, linetype = "dashed", color = "black") +
    geom_vline(xintercept = 0, linetype = "dashed", color = "black") +
    ggrepel::geom_text_repel(show.legend = FALSE, color = "black") +
    theme_bw(base_size = 15) +
    labs(x = "logFC by MiloR", y = "AUC score by Augur", title = plot_title) +
    theme(legend.position = "none", panel.grid.major = element_blank(), panel.grid.minor = element_blank())
  ggsave(file.path(out_dir, paste0("Combined_", ct_level, "_", out_prefix, ".pdf")),
         p_combined, width = 8, height = 7)
}

## 主循环：3 个临床对比 × H2/H3
for (cname in names(contrasts)) {
  conf <- contrasts[[cname]]
  message("==== contrast: ", conf$tag, " ====")
  ginfo <- conf$make_group(tmp)
  names(ginfo$group) <- ginfo$cells
  obj_use <- subset(tmp, cells = ginfo$cells)
  obj_use$compare_group <- ginfo$group[colnames(obj_use)]
  message("  cells: ", ncol(obj_use))
  print(table(obj_use$compare_group, useNA = "ifany"))

  for (ct_col in ct_cols) {
    ct_level <- ifelse(grepl("H2$", ct_col), "H2", "H3")
    out_prefix <- paste0("NaivePC_", conf$tag)
    plot_title <- paste0("Naive_PC | ", conf$tag, " | ", ct_col)
    message("-- ", plot_title)

    tryCatch({
      run_milo_augur(
        obj = obj_use,
        group_vec = factor(obj_use$compare_group, levels = levels(ginfo$group)),
        ct_col = ct_col,
        out_prefix = out_prefix,
        plot_title = plot_title
      )
      message("  done: ", out_prefix, " / ", ct_level)
    }, error = function(e) {
      message("  FAILED: ", out_prefix, " / ", ct_level, " -> ", conditionMessage(e))
    })
  }
}

message("All finished. PDFs in: ", out_dir)

Loading required package: edgeR

Loading required package: limma




Attaching package: ‘limma’


The following object is masked from ‘package:BiocGenerics’:

    plotMA



Attaching package: ‘edgeR’


The following object is masked from ‘package:SingleCellExperiment’:

    cpm


Loading required package: scuttle


Attaching package: ‘scuttle’


The following object is masked from ‘package:celda’:

    normalizeCounts



Attaching package: ‘scater’


The following object is masked from ‘package:limma’:

    plotMDS


The following object is masked from ‘package:celda’:

    plotHeatmap


Registered S3 method overwritten by 'yardstick':
  method       from         
  print.metric spatstat.geom

==== contrast: PSA ====

  cells: 87334




 PSA_low PSA_high 
   73430    13904 


-- Naive_PC | PSA | celltype_manual_H2

Constructing kNN graph with k:30

Checking valid object

Running refined sampling with reduced_dim

Checking meta.data validity

Counting cells in neighbourhoods

'as(<dgTMatrix>, "dgCMatrix")' is deprecated.
Use 'as(., "CsparseMatrix")' instead.
See help("Deprecated") and help("Matrix-deprecated").

Using TMM normalisation

Running with model contrasts

Performing spatial FDR correction with k-distance weighting

Converting celltype to factor...

Converting group_by to factor...

Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.
using default assay: RNA ...

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `metrics = pred %>% map(metric_fun)`.
Caused by warning:
! While computing binary `precision()`, no predicted events were detected (i.e.
`true_positive + false_positive = 0`).
Precision is undefined in this case, and `NA` will be returned.
Note that 7 true event(s) a


stage_pT2 stage_pT3 
    44133     43201 


-- Naive_PC | stage | celltype_manual_H2

Constructing kNN graph with k:30

Checking valid object

Running refined sampling with reduced_dim

Checking meta.data validity

Counting cells in neighbourhoods

Using TMM normalisation

Running with model contrasts

Performing spatial FDR correction with k-distance weighting

Converting celltype to factor...

Converting group_by to factor...

Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.
using default assay: RNA ...

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `metrics = pred %>% map(metric_fun)`.
Caused by warning:
! While computing binary `precision()`, no predicted events were detected (i.e.
`true_positive + false_positive = 0`).
Precision is undefined in this case, and `NA` will be returned.
Note that 7 true event(s) actually occurred for the problematic event level,
stage_pT2”
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `metrics


 Gleason_low Gleason_high 
       49330        29547 


-- Naive_PC | Gleason | celltype_manual_H2

Constructing kNN graph with k:30

Checking valid object

Running refined sampling with reduced_dim

Checking meta.data validity

Counting cells in neighbourhoods

Using TMM normalisation

Running with model contrasts

Performing spatial FDR correction with k-distance weighting

Converting celltype to factor...

Converting group_by to factor...

Scale for colour is already present.
Adding another scale for colour, which will replace the existing scale.
using default assay: RNA ...

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `metrics = pred %>% map(metric_fun)`.
Caused by warning:
! While computing binary `precision()`, no predicted events were detected (i.e.
`true_positive + false_positive = 0`).
Precision is undefined in this case, and `NA` will be returned.
Note that 7 true event(s) actually occurred for the problematic event level,
Gleason_high”
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `me